In [1]:
import generate_graph

In [2]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve().parents[1]
print(project_root)
sys.path.insert(0, str(project_root))

import lib.dataloader as dl

/vol/data/immuneML/cmsb26_project7


In [17]:
import itertools

seeds = [3,5,7]
seeds = [3]

freqs = ["010","025","050"]

sizes = [50,100,150,200,1000]
sizes = [200]

noise = [5,25,50,60,70,80]

datasets = []

# generate all possible datasets
for s,f,sz,n in itertools.product(seeds,freqs,sizes,noise):
    name = f"repertoire_LigoSim_final_testing_variant_seed{s}_freq{f}_size{sz}_noise{n}"
    datasets.append(name)
print(datasets)

['repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise5', 'repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise25', 'repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise50', 'repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise60', 'repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise70', 'repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise80', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise5', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise25', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise50', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise60', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise70', 'repertoire_LigoSim_final_testing_variant_seed3_freq025_size200_noise80', 'repertoire_LigoSim_final_testing_variant_seed3_freq050_size200_noise5', 'repertoire_LigoSim_final_testing_varian

In [5]:
for dataset in datasets[:11]:
    df = dl.load_airr_dataset(dataset)


Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise5/my_sim_inst/exported_dataset/airr/_dataset.pkl...
Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise25/my_sim_inst/exported_dataset/airr/_dataset.pkl...
Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise50/my_sim_inst/exported_dataset/airr/_dataset.pkl...
Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise60/my_sim_inst/exported_dataset/airr/_dataset.pkl...
Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise70/my_sim_inst/exported_dataset/airr/_dataset.pkl...
Loading cached dataset from /vol/data/immuneML/simulatio

In [6]:
import numpy as np
import encoding.tcr_bert as tb

def process_dataset(dataset):
    df = dl.load_airr_dataset(dataset)
    df["templates"] = 1
    df = df[
        [
            "junction_aa",
            "templates",
            "filename",
            "disease",
            "sample",
            "repertoire_id",
        ]
    ]

    df = generate_graph.top_percent_cutoff(df, p=0.01)
    df = df.reset_index(drop=True)
    df = df.dropna()

    df_enc = tb.encode_sequences(df, seq_col="junction_aa", max_length=30)
    _, _, embeddings = df_enc

    df["label_positive"] = df["disease"]
    df["age"] = 'N'
    df["sex"] = 'N'
    df["study_group_description"] = 'N'

    return df, embeddings

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/230M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: wukevin/tcr-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/230M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/91.0 [00:00<?, ?B/s]

TCR-BERT loaded on cpu


In [7]:
from sklearn.neighbors import kneighbors_graph
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.spatial.distance import cosine as cosine_distance


def build_knn_tcr_graph(cluster_embeddings, frequencies, k=10):
    num_clusters = len(cluster_embeddings)
    
    # Node features (with frequency)
    freq_normalized = (frequencies / frequencies.sum()).reshape(-1, 1)
    node_features = torch.cat([
        torch.tensor(cluster_embeddings, dtype=torch.float),
        torch.tensor(freq_normalized, dtype=torch.float)
    ], dim=1)
    
    # Create KNN graph
    knn = kneighbors_graph(cluster_embeddings, k, mode='connectivity', 
                          include_self=False, metric='cosine')
    
    # Convert to edge_index format
    edges = []
    edge_weights = []
    
    for i in range(num_clusters):
        neighbors = knn[i].nonzero()[1]
        for j in neighbors:
            edges.append([i, j])
            # Calculate similarity for edge weight
            sim = 1 - cosine_distance(cluster_embeddings[i], cluster_embeddings[j])
            edge_weights.append(sim)
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_weights, dtype=torch.float).unsqueeze(1)
    
    return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)

def build_graphs_for_samples(df, k=10):
    graphs = {}
    for sample_id, group in df.groupby("sample"):
        cluster_embeddings = np.array(group["cluster_embedding"].tolist())
        frequencies = group["templates"].values
        graph = build_knn_tcr_graph(cluster_embeddings, frequencies, k)
        # graph label either 0 or 1 based on bool in label_positive column
        graph.y = torch.tensor([1 if group["label_positive"].iloc[0] else 0], dtype=torch.long)
        graphs[sample_id] = graph
    return graphs

In [8]:
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

def split_data(dataset):

    train_data, val_data = train_test_split(
        dataset,
        test_size=0.2,
        random_state=42,
        stratify=[data.y.item() for data in dataset]  # keeps class balance
    )

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_data, batch_size=16, shuffle=False)

    return train_data, val_data, train_loader, val_loader

In [ ]:
import tcrGNN
import torch
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix


def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch).view(-1)
        y = batch.y.view(-1).float()

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            out = model(batch).view(-1)
            y = batch.y.view(-1).float()

            pred = (torch.sigmoid(out) > 0.5).float()

            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total


def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            out = model(batch).view(-1)
            y = batch.y.view(-1).float()

            loss = criterion(out, y)
            total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_auc(model, loader, device):
    model.eval()

    probs = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            out = model(batch).view(-1)
            p = torch.sigmoid(out)

            probs.extend(p.cpu().numpy())
            labels.extend(batch.y.view(-1).cpu().numpy())

    auc = roc_auc_score(labels, probs)

    return auc, labels, probs


def train_model(train_data, val_data, train_loader, val_loader,
                num_epochs=100, print_every=20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    in_dim = train_data[0].x.shape[1]
    hidden_dim = 16
    num_classes = 1

    model = tcrGNN.TCRgnnEdge(in_dim, hidden_dim, num_classes).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    train_labels = torch.tensor([data.y.item() for data in train_data])
    num_pos = (train_labels == 1).sum().item()
    num_neg = (train_labels == 0).sum().item()

    pos_weight = torch.tensor(num_neg / num_pos).to(device)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(num_epochs):

        train_loss = train(model, train_loader, optimizer, criterion, device)
        train_acc  = evaluate(model, train_loader, device)

        val_loss = evaluate_loss(model, val_loader, criterion, device)
        val_acc  = evaluate(model, val_loader, device)

        val_auc, labels, probs = evaluate_auc(model, val_loader, device)
        preds = (torch.tensor(probs) >= 0.5).int().numpy()

        precision = precision_score(labels, preds, zero_division=0)
        recall = recall_score(labels, preds, zero_division=0)
        cm = confusion_matrix(labels, preds)

        if (epoch + 1) % print_every == 0:
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"Train Loss: {train_loss:.4f}")
            print(f"Val Loss:   {val_loss:.4f}")
            print(f"Train Acc:  {train_acc:.4f}")
            print(f"Val Acc:    {val_acc:.4f}")
            print(f"Val AUC:    {val_auc:.4f}")
            print(f"Precision:  {precision:.4f}")
            print(f"Recall:     {recall:.4f}")
            print("Confusion Matrix:")
            print(cm)

    return {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "val_auc": val_auc,
        "precision": precision,
        "recall": recall,
        "confusion_matrix": cm
    }

In [18]:
for dataset in datasets[:10]:
    print(dataset)
    df, embeddings = process_dataset(dataset)

    samples = df["sample"].unique()

    print(len(samples))

    df["cluster_embedding"] = list(embeddings)

    graphs_dict = build_graphs_for_samples(df, k=len(samples)//10)
    dataset = list(graphs_dict.values())

    train_data, val_data, train_loader, val_loader = split_data(dataset)

    train_model(train_data, val_data, train_loader, val_loader)

repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise5
Loading cached dataset from /vol/data/immuneML/simulation_data/data/repertoire_LigoSim_final_testing_variant_seed3_freq010_size200_noise5/my_sim_inst/exported_dataset/airr/_dataset.pkl...
['CASGPKRSSGGGTDTQYF', 'CSARGPKPEGSPLHF', 'CASSFPRQGGPKTDTQYF', 'CASSYLGAGPKYNEQFF', 'CASSSGEGGPKF', 'CASSAGGEIGPKYTF', 'CASGPKDLGQLSYEQYF', 'CASSQAQGLNTGPKFF', 'CASSLSNETGQRATDTGPK', 'CASSWKSGSIGASNQPGPK', 'CASSWGKGRGLRTDTQYF', 'CASSIIVRGIQNTEAFF', 'CASSLAWGPRNQPQHF', 'CASSLARGAYEQYF', 'CASSQAVLYEKLFF', 'CASSEETRGANTGELFF', 'CASSLDLNYNYGYTF', 'CASRGRGRPLHF', 'CATRVQAKGELFF', 'CASSLTGRAIIQYF']
['C A S G P K R S S G G G T D T Q Y F', 'C S A R G P K P E G S P L H F', 'C A S S F P R Q G G P K T D T Q Y F', 'C A S S Y L G A G P K Y N E Q F F', 'C A S S S G E G G P K F', 'C A S S A G G E I G P K Y T F', 'C A S G P K D L G Q L S Y E Q Y F', 'C A S S Q A Q G L N T G P K F F', 'C A S S L S N E T G Q R A T D T G P K', 'C A S S W K S G S I G 

Encoding sequences: 100%|██████████| 20/20 [02:22<00:00,  7.10s/it]


200

Epoch 20/100
Train Loss: 0.6963
Val Loss:   0.6897
Train Acc:  0.5000
Val Acc:    0.5000
Val AUC:    0.3950
Precision:  0.0000
Recall:     0.0000
Confusion Matrix:
[[20  0]
 [20  0]]

Epoch 40/100
Train Loss: 0.6959
Val Loss:   0.6919
Train Acc:  0.5000
Val Acc:    0.5000
Val AUC:    0.5450
Precision:  0.0000
Recall:     0.0000
Confusion Matrix:
[[20  0]
 [20  0]]

Epoch 60/100
Train Loss: 0.6965
Val Loss:   0.6892
Train Acc:  0.5000
Val Acc:    0.5000
Val AUC:    0.5525
Precision:  0.0000
Recall:     0.0000
Confusion Matrix:
[[20  0]
 [20  0]]


KeyboardInterrupt: 